# Neural network training

This notebook trains the final three-class network used for autonomous navigation.
The training set contains the original labelled camera images together with manually reviewed hard examples collected during live validation.
Upload dataset.zip when requested and run all cells in order.


In [ ]:
from pathlib import Path
from google.colab import files

DATASET_ZIP = Path('/content/dataset.zip')

if not DATASET_ZIP.exists():
    print('Upload dataset.zip')
    uploaded = files.upload()
    if 'dataset.zip' not in uploaded:
        raise FileNotFoundError('dataset.zip was not uploaded')


In [ ]:
import re
import zipfile
import cv2
import numpy as np
import tensorflow as tf

LABELS = ('forward', 'left', 'right')
EPOCHS = 3200
LEARNING_RATE = 0.01
L2 = 1e-4
SEED = 42

def prepare(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    return gray.astype(np.float32).reshape(-1) / 255.0

records = {'official': [], 'old': [], 'new': []}

with zipfile.ZipFile(DATASET_ZIP) as archive:
    for name in sorted(x for x in archive.namelist() if x.lower().endswith('.jpg')):
        label = name.split('/')[0]
        filename = Path(name).name
        image = cv2.imdecode(
            np.frombuffer(archive.read(name), dtype=np.uint8),
            cv2.IMREAD_COLOR
        )
        if image is None or image.shape != (50, 50, 3):
            raise ValueError(f'Unexpected image: {name}')
        row = (filename, LABELS.index(label), prepare(image), label)
        if filename.startswith('oldhard'):
            records['old'].append(row)
        elif filename.startswith('newhard'):
            records['new'].append(row)
        else:
            records['official'].append(row)

def pulse_number(filename):
    return int(re.search(r'(\d+)', filename).group(1))

def arrays(rows):
    x = np.stack([row[2] for row in rows])
    y = np.asarray([row[1] for row in rows], dtype=np.int32)
    return x, y

official = records['official']
old = sorted(records['old'], key=lambda row: pulse_number(row[0]))
new = sorted(records['new'], key=lambda row: pulse_number(row[0]))

special = {145, 147, 153}
base = [row for row in new if pulse_number(row[0]) not in special]
baseforward = [row for row in base if row[3] == 'forward']
baseleft = [row for row in base if row[3] == 'left']
newbypulse = {pulse_number(row[0]): row for row in new}

blocks = []

for rows, weight in [
    (official, 1.0),
    (old, 3.0),
    (baseforward, 1.0),
    (baseleft, 2.4),
]:
    x, y = arrays(rows)
    blocks.append((x, y, weight))

for pulse, weight in [(145, 2.0), (147, 2.8), (153, 2.0)]:
    row = newbypulse[pulse]
    blocks.append((
        np.stack([row[2]]),
        np.asarray([row[1]], dtype=np.int32),
        weight
    ))

x = np.concatenate([block[0] for block in blocks])
y = np.concatenate([block[1] for block in blocks])
sampleweights = np.concatenate([
    np.full(len(block[1]), block[2], dtype=np.float32)
    for block in blocks
])

print('Training images:', len(y))
print('Forward:', int(np.sum(y == 0)))
print('Left:', int(np.sum(y == 1)))
print('Right:', int(np.sum(y == 2)))


In [ ]:
rng = np.random.default_rng(SEED)
weights = rng.normal(0, 0.01, (2500, 3)).astype(np.float32)
bias = np.zeros(3, dtype=np.float32)

targets = np.eye(3, dtype=np.float32)[y]
denominator = float(sampleweights.sum())

for epoch in range(EPOCHS):
    logits = x @ weights + bias
    logits -= logits.max(axis=1, keepdims=True)
    expvalues = np.exp(logits)
    probabilities = expvalues / expvalues.sum(axis=1, keepdims=True)

    gradient = ((probabilities - targets) * sampleweights[:, None]) / denominator

    weights -= LEARNING_RATE * (
        x.T @ gradient + L2 * weights
    ).astype(np.float32)

    bias -= LEARNING_RATE * gradient.sum(axis=0).astype(np.float32)

predictions = np.argmax(x @ weights + bias, axis=1)
accuracy = float(np.mean(predictions == y))
print('Training accuracy:', accuracy)


In [ ]:
model = tf.keras.Sequential([
    tf.keras.Input(shape=(2500,), dtype='float32'),
    tf.keras.layers.Dense(3, activation='softmax', name='direction')
])

model.get_layer('direction').set_weights([weights, bias])

networkfile = Path('/content/network.h5')
model.save(networkfile)

reloaded = tf.keras.models.load_model(networkfile, compile=False)
testoutput = reloaded(x[:10], training=False).numpy()
originaloutput = model(x[:10], training=False).numpy()
np.testing.assert_allclose(testoutput, originaloutput, rtol=1e-6, atol=1e-7)

print('Saved:', networkfile)
files.download(str(networkfile))
